#  Smart Finance AI Assistant

### Introduction to Programming Project

An AI-powered Telegram bot for:
-  Income tracking
-  Expense management
-  Financial analytics
-  Smart calculations
-  AI financial advice

---

### Technologies Used
- Python
- Telegram Bot API
- JSON storage
- Regular Expressions
- Natural Language Processing Logic

## Installing Required Libraries

This section installs the Telegram Bot API package.

In [1]:
!pip install pyTelegramBotAPI

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.3/48.3 kB 1.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 308.4/308.4 kB 2.8 MB/s eta 0:00:00


## 💾 Data Storage System

The bot stores user financial data using a JSON file.
Functions:
- load_data()
- save_data()

In [2]:
import json
from datetime import datetime

DATA_FILE = "data.json"

def load_data():
    try:
        with open(DATA_FILE, "r") as f:
            return json.load(f)
    except:
        return {}

def save_data(data):
    with open(DATA_FILE, "w") as f:
        json.dump(data, f, indent=4)

##  Main AI Bot Engine

Main features:
- Natural language understanding
- Income & expense tracking
- Financial analytics
- Smart calculator
- AI financial advice
- Multi-command processing

In [ ]:
import telebot
import re

TOKEN = "8781923832:AAHvotJZHljE-OH2f54BTaIMriUZO78Q9e8"
bot = telebot.TeleBot(TOKEN)

# ================= UTIL =================
def get_number(text):
    nums = re.findall(r'\d+', text)
    return int(nums[0]) if nums else None


def is_income(text):
    keywords = ["income", "salary", "earned", "got", "receive", "доход", "зарплата"]
    return any(k in text.lower() for k in keywords)


def is_expense(text):
    keywords = ["spent", "paid", "cost", "buy", "bought", "расход", "потратил"]
    return any(k in text.lower() for k in keywords)


# ================= START (PREMIUM UI) =================
@bot.message_handler(commands=['start'])
def start(message):
    bot.send_message(message.chat.id,
        "💎 SMART FINANCE AI ASSISTANT\n"
        "━━━━━━━━━━━━━━━━━━━━━━\n\n"
        "🤖 Personal financial tracker + AI calculator\n"
        "🧠 Understands natural language (no strict format)\n\n"
        "✨ WHAT YOU CAN DO:\n"
        "💰 Track income\n"
        "💸 Track expenses\n"
        "📊 View balance & reports\n"
        "🧮 Calculate percentages\n\n"
        "━━━━━━━━━━━━━━━━━━━━━━\n"
        "💬 EXAMPLES:\n\n"
        "• my income is 50000\n"
        "• I spent 12000 rent\n"
        "• 50% of 50000\n"
        "• show balance\n"
        "• show table\n"
        "• I earned 50000 and spent 10000 and show balance\n\n"
        "━━━━━━━━━━━━━━━━━━━━━━\n"
        "🚀 TIP: You can write everything in one sentence"
    )


# ================= RESET =================
@bot.message_handler(commands=['reset'])
def reset(message):
    data = load_data()
    user_id = str(message.chat.id)

    data[user_id] = []
    save_data(data)

    bot.send_message(message.chat.id, "🧹 All data successfully cleared.")


# ================= HELP =================
@bot.message_handler(commands=['help'])
def help_cmd(message):
    bot.send_message(message.chat.id,
        "📌 SMART FINANCE AI GUIDE\n\n"
        "💰 Income:\n"
        "- my income is 50000\n\n"
        "💸 Expense:\n"
        "- I spent 10000 food\n\n"
        "🧮 Calculator:\n"
        "- 20% of 50000\n\n"
        "📊 Reports:\n"
        "- show balance\n"
        "- show table\n\n"
        "🔥 Multi-input supported:\n"
        "- I earned 50000 and spent 10000 rent show balance"
    )


# ================= MAIN ENGINE =================
@bot.message_handler(func=lambda message: True)
def brain(message):

    text = message.text.lower()

    user_id = str(message.chat.id)

    data = load_data()

    if user_id not in data:
        data[user_id] = []

    # ================= SPLIT MULTI COMMAND =================
    parts = re.split(r'and|,|\n', text)

    for part in parts:
        part = part.strip()

        # ================= RESET =================
        if "reset" in part:
            data[user_id] = []
            save_data(data)
            bot.send_message(message.chat.id, "🧹 Data reset completed")
            continue

        # ================= BALANCE =================
        if "balance" in part:

            income = sum(i["amount"] for i in data[user_id] if i["type"] == "income")
            expense = sum(i["amount"] for i in data[user_id] if i["type"] == "expense")

            bot.send_message(message.chat.id,
                f"📊 FINANCIAL SUMMARY\n\n"
                f"💰 Income: {income}\n"
                f"💸 Expenses: {expense}\n"
                f"📉 Balance: {income - expense}"
            )
            continue

        # ================= TABLE =================
        if "table" in part or "show" in part:

            rows = data[user_id]

            if not rows:
                bot.send_message(message.chat.id, "📭 No financial records yet.")
                continue

            income = sum(i["amount"] for i in rows if i["type"] == "income")
            expense = sum(i["amount"] for i in rows if i["type"] == "expense")

            msg = "📊 TRANSACTION HISTORY\n"
            msg += "━━━━━━━━━━━━━━\n"

            for r in rows:
                msg += f"• {r['type'].upper()} → {r['amount']}\n"

            msg += "━━━━━━━━━━━━━━\n"
            msg += f"💰 Income: {income}\n"
            msg += f"💸 Expenses: {expense}\n"
            msg += f"📉 Balance: {income - expense}"

            bot.send_message(message.chat.id, msg)
            continue

        # ================= INCOME =================
        if is_income(part) and get_number(part):

            amount = get_number(part)

            data[user_id].append({"type": "income", "amount": amount})
            save_data(data)

            bot.send_message(message.chat.id, f"💰 Income recorded: {amount}")
            continue

        # ================= EXPENSE =================
        if is_expense(part) and get_number(part):

            amount = get_number(part)

            data[user_id].append({"type": "expense", "amount": amount})
            save_data(data)

            bot.send_message(message.chat.id, f"💸 Expense recorded: {amount}")

            continue
                    # ================= AI ADVICE =================
        if "advice" in part or "tips" in part or "save money" in part:

            income = sum(i["amount"] for i in data[user_id] if i["type"] == "income")
            expense = sum(i["amount"] for i in data[user_id] if i["type"] == "expense")

            balance = income - expense

            if income == 0:
                msg = (
                    "🧠 AI FINANCIAL ADVICE\n\n"
                    "💡 Start by tracking your income first."
                )

            elif expense > income:
                msg = (
                    "🧠 AI FINANCIAL ADVICE\n\n"
                    "⚠️ Your expenses are higher than your income.\n"
                    "Try reducing non-essential spending."
                )

            elif expense > income * 0.7:
                msg = (
                    "🧠 AI FINANCIAL ADVICE\n\n"
                    "💡 You are spending more than 70% of your income.\n"
                    "Recommendation:\n"
                    "• reduce unnecessary purchases\n"
                    "• save at least 20% monthly"
                )

            else:
                msg = (
                    "🧠 AI FINANCIAL ADVICE\n\n"
                    "🔥 Excellent financial control.\n"
                    "You are managing your money wisely."
                )

            msg += f"\n\n📉 Current balance: {balance}"

            bot.send_message(message.chat.id, msg)
            continue

        # ================= CALCULATOR =================
        if "%" in part:

            nums = re.findall(r'\d+', part)

            if len(nums) >= 2:
                percent = int(nums[0])
                value = int(nums[1])

                result = value * percent / 100

                bot.send_message(message.chat.id,
                    f"🧮 CALCULATION RESULT\n\n"
                    f"{percent}% of {value} = {result}"
                )
                continue


print("💎 SMART FINANCE AI IS RUNNING...")
bot.infinity_polling(skip_pending=True)

💎 SMART FINANCE AI IS RUNNING...


#  Project Summary

The Smart Finance AI Assistant is a Telegram-based financial chatbot that helps users manage personal finances through natural language interaction.

Key Features:
- AI-style conversation
- Financial tracking
- Percentage calculations
- Balance analytics
- Smart financial recommendations

This project demonstrates:
- Python programming
- Functions & modularity
- Data structures
- File handling
- NLP-inspired logic
- Telegram Bot integration